# Regime-adaptive XGBoost

This directional backtest combines XGBoost with a hidden Markov model and recalibrates entry and exit thresholds across walk-forward windows. It uses NVDA as the main asset, QQQ and SPY for cross-asset features, and URTH as a benchmark.

The notebook downloads daily split- and dividend-adjusted prices from Yahoo Finance with `yfinance`. It saves charts and metrics to `results/xgboost-regime-adaptive/`.

The final step repeats the walk-forward backtest for 14 stocks and can take several minutes.


In [ ]:
"""
================================================================================
ML REGIME-ADAPTIVE TRADING RESEARCH V3
WITH WFA-OPTIMIZED HYSTERESIS + MULTI-TICKER VALIDATION
================================================================================

V3 CONSOLIDATION - All improvements from V2 + Multi-Ticker Validation:

1. WFA-Integrated Hysteresis Optimization
   - Entry/exit thresholds optimized per fold on validation set
   - Uses DAILY returns (not overlapping n-day) to avoid corrupted Sharpe
   - Minimum holding period enforced to match prediction horizon
   - Parameter Stability Index (PSI) tracks threshold variance

2. Bug Fixes (from V2):
   - Regime features shifted with apply_shift=True (no look-ahead)
   - CVaR stop handles both long AND short positions correctly
   - Symmetric hysteresis for long/short entries
   - Feature warm-up validation before backtest start
   - Cash reserve properly updated in non-compounding mode
   - sklearn StandardScaler for HMM (not manual implementation)

3. Return Enhancements (No Overfitting Risk):
   - Volatility-scaled position sizing (target vol)
   - Regime-aware allocation multipliers
   - Probability-weighted sizing
   - 2x embargo for target leakage prevention

4. Multi-Ticker Validation (preserved):
   - Cross-validation across 14 tickers
   - GRI > 0.60 threshold, 70% success rate criteria
   - Detailed results saved to JSON/CSV

5. Feature Selection (preserved):
   - Dynamic selection on first fold
   - Top N features by importance
   - Handles dynamically added features (regime)
"""

from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import t, norm
from hmmlearn import hmm
import matplotlib.pyplot as plt
import json
import warnings

from _market_data import download_ohlcv
warnings.filterwarnings('ignore')


# ==============================================================================
# CONFIGURATION CLASS
# ==============================================================================
class StrategyConfig:
    """
    Centralized configuration - all parameters defined here.
    
    V3: Added hysteresis optimization grid and position sizing enhancements.
    """
    
    # ============ DATA PARAMETERS ============
    BACKTEST_START = "2018-12-31"
    BACKTEST_END = "2023-12-31"
    WARMUP_MONTHS = 24
    
    SYMBOL_MAIN = "NVDA"
    SYMBOLS_CORR = ["QQQ", "SPY"]
    BENCHMARK = "URTH"
    
    @property
    def START_DATE(self):
        start = pd.to_datetime(self.BACKTEST_START)
        return (start - pd.DateOffset(months=self.WARMUP_MONTHS)).strftime("%Y-%m-%d")
    
    @property  
    def END_DATE(self):
        return self.BACKTEST_END
    
    # ============ FEATURE PARAMETERS ============
    RETURN_WINDOWS = [5, 10, 20, 60]
    YZRV_WINDOW = 20
    HAR_WINDOWS = {'daily': 1, 'weekly': 5, 'monthly': 22}
    CORR_WINDOWS = [20, 60]
    MOMENTUM_WINDOWS = [20, 60, 120]
    TSMOM_WINDOWS = [20, 60, 120]
    ENTROPY_WINDOW = 20
    ENTROPY_BINS = 10
    HURST_WINDOWS = [60, 120]
    SMA_WINDOWS = [50, 200]
    AMIHUD_WINDOW = 20
    ATR_WINDOW = 14
    MAX_FEATURE_LOOKBACK = 126
    
    # ============ HMM REGIME PARAMETERS ============
    N_REGIMES = 3
    HMM_COVARIANCE = 'full'
    HMM_N_ITER = 100
    HMM_RANDOM_STATE = 42
    
    # ============ XGBOOST PARAMETERS ============
    # V3 UPDATE: Using V2 params (300 trees, depth 4) for better alpha capture
    XGB_PARAMS = {
        'n_estimators': 300,
        'max_depth': 4,
        'learning_rate': 0.03,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
        'min_child_weight': 3,
        'gamma': 0.1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'random_state': 42,
        'n_jobs': -1
    }
    
    # ============ WALK-FORWARD PARAMETERS ============
    VOL_NORMALIZATION = True
    VOL_LOOKBACK = 20
    TRAIN_RATIO = 0.60
    VALIDATE_RATIO = 0.20
    TEST_RATIO = 0.20
    REFIT_FREQUENCY = 20
    PREDICTION_HORIZON = 5
    MIN_TRAIN_DAYS = 252
    EMBARGO_MULTIPLIER = 2  # 2x horizon for safety
    
    # ============ FEATURE SELECTION ============
    USE_FEATURE_SELECTION = True
    N_FEATURES_SELECT = 15
    
    # ============ HYSTERESIS OPTIMIZATION (V2) ============
    USE_HYSTERESIS = True
    OPTIMIZE_HYSTERESIS = True
    ENTRY_THRESHOLD_GRID = [0.55, 0.58, 0.60, 0.62, 0.65, 0.68, 0.70]
    EXIT_THRESHOLD_GRID = [0.45, 0.48, 0.50, 0.52]
    ENTRY_THRESHOLD = 0.60  # Default if optimization disabled
    EXIT_THRESHOLD = 0.50
    
    # ============ POSITION SIZING ENHANCEMENTS (V2) ============
    USE_VOL_SCALING = True
    TARGET_VOLATILITY = 0.20
    VOL_SCALE_CAP = 2.0
    
    USE_REGIME_ALLOCATION = True
    REGIME_MULTIPLIERS = {0: 1.0, 1: 0.7, 2: 0.4}
    
    USE_PROBABILITY_SIZING = True
    MIN_CONFIDENCE_ALLOCATION = 0.5
    
    # ============ CHANDELIER STOP ============
    USE_CHANDELIER_STOP = True
    ATR_MULTIPLIER = 3.0
    
    # ============ CVAR STOP ============
    USE_CVAR_STOP = True
    CVAR_LOOKBACK = 60
    CVAR_QUANTILE = 0.95
    CVAR_MULTIPLIER = 1.5
    
    # ============ BACKTEST PARAMETERS ============
    INITIAL_CAPITAL = 100000.0
    COMPOUNDING_MODE = True
    LEVERAGE = 1.0
    COMMISSION_PCT = 0.0005
    SLIPPAGE_BPS = 5
    MAX_POSITION_SIZE = 1.0
    
    # ============ ROBUSTNESS METRICS ============
    N_TESTS_ADJUSTMENT = 100
    CONFIDENCE_LEVEL = 0.95
    CDAR_QUANTILE = 0.95
    
    # ============ VISUALIZATION ============
    PLOT_FIGSIZE = (16, 12)
    PLOT_DPI = 100


# ==============================================================================
# DATA FETCHING MODULE
# ==============================================================================
def fetch_ohlcv(symbol, start_date, end_date):
    """Fetch daily OHLCV with split and dividend adjustments from Yahoo Finance."""
    try:
        prices = download_ohlcv(symbol, start_date, end_date, auto_adjust=True)
    except RuntimeError as error:
        print(f"[ERROR] {error}")
        return pd.DataFrame()

    print(f"[OK] Fetched {len(prices)} rows for {symbol}")
    return prices.rename(columns=str.lower)[["open", "high", "low", "close", "volume"]]


# ==============================================================================
# FEATURE ENGINEERING MODULE
# ==============================================================================
class FeatureEngine:
    """Comprehensive feature engineering with temporal controls."""
    
    def __init__(self, config):
        self.config = config
    
    def compute_basic_returns(self, df):
        df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
        df["ooc_ret"] = np.log(df["close"] / df["open"])
        df["ooo_ret"] = np.log(df["open"] / df["open"].shift(1))
        return df
    
    def compute_rolling_returns(self, df):
        for w in self.config.RETURN_WINDOWS:
            df[f"log_ret_roll_{w}"] = df["log_ret"].rolling(w).sum()
            df[f"ooc_roll_{w}"] = df["ooc_ret"].rolling(w).sum()
            df[f"ooo_roll_{w}"] = df["ooo_ret"].rolling(w).sum()
        return df
    
    def compute_yang_zhang_volatility(self, df):
        window = self.config.YZRV_WINDOW
        ro = np.log(df["open"] / df["close"].shift(1))
        rc = np.log(df["close"] / df["open"])
        rs = np.log(df["high"] / df["low"])
        
        sigma_o = ro.rolling(window).var()
        sigma_c = rc.rolling(window).var()
        sigma_rs = (rs ** 2).rolling(window).mean()
        
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        df["yzrv"] = np.sqrt(sigma_o + k * sigma_c + (1 - k) * sigma_rs)
        return df
    
    def compute_har_structure(self, df):
        df["yzrv_d"] = df["yzrv"]
        df["yzrv_w"] = df["yzrv"].rolling(self.config.HAR_WINDOWS['weekly']).mean()
        df["yzrv_m"] = df["yzrv"].rolling(self.config.HAR_WINDOWS['monthly']).mean()
        df["har_signal"] = df["yzrv_d"] + df["yzrv_w"] + df["yzrv_m"]
        return df
    
    def compute_cross_correlations(self, df, corr_assets):
        main_ret = df["log_ret"]
        for asset_name, asset_df in corr_assets.items():
            asset_ret = np.log(asset_df["close"] / asset_df["close"].shift(1))
            df[f"{asset_name}_ret"] = asset_ret
            for w in self.config.CORR_WINDOWS:
                df[f"corr_{asset_name}_{w}"] = main_ret.rolling(w).corr(df[f"{asset_name}_ret"])
        return df
    
    def compute_amihud_illiquidity(self, df):
        df["amihud"] = (df["log_ret"].abs() / df["volume"]).rolling(self.config.AMIHUD_WINDOW).mean()
        return df
    
    def compute_momentum(self, df):
        for w in self.config.MOMENTUM_WINDOWS:
            df[f"momentum_{w}"] = df["close"] / df["close"].shift(w) - 1
        return df
    
    def compute_tsmom(self, df):
        for w in self.config.TSMOM_WINDOWS:
            df[f"tsmom_{w}"] = np.sign(df["log_ret"].rolling(w).mean())
        return df
    
    def compute_entropy(self, df):
        window = self.config.ENTROPY_WINDOW
        n_bins = self.config.ENTROPY_BINS
        
        def rolling_entropy(x):
            if len(x) < window or x.isna().all():
                return np.nan
            x_clean = x.dropna()
            if len(x_clean) < 5:
                return np.nan
            counts, _ = np.histogram(x_clean, bins=n_bins)
            probs = counts / counts.sum()
            probs = probs[probs > 0]
            return -np.sum(probs * np.log(probs))
        
        df["entropy"] = df["log_ret"].rolling(window).apply(rolling_entropy, raw=False)
        return df
    
    def compute_hurst_exponent(self, df):
        def hurst_rs(x):
            if len(x) < 20 or x.isna().any():
                return np.nan
            y = x - x.mean()
            z = np.cumsum(y)
            R = z.max() - z.min()
            S = x.std()
            if S == 0 or R == 0:
                return np.nan
            return np.log(R / S) / np.log(len(x))
        
        for w in self.config.HURST_WINDOWS:
            df[f"hurst_{w}"] = df["log_ret"].rolling(w).apply(hurst_rs, raw=False)
        return df
    
    def compute_sma_features(self, df):
        for w in self.config.SMA_WINDOWS:
            df[f"sma_{w}"] = df["close"].rolling(w).mean()
            df[f"price_vs_sma_{w}"] = (df["close"] - df[f"sma_{w}"]) / df[f"sma_{w}"]
            df[f"sma_{w}_slope"] = df[f"sma_{w}"] / df[f"sma_{w}"].shift(5) - 1
        return df
    
    def compute_atr(self, df):
        window = self.config.ATR_WINDOW
        high_low = df['high'] - df['low']
        high_close_prev = (df['high'] - df['close'].shift(1)).abs()
        low_close_prev = (df['low'] - df['close'].shift(1)).abs()
        true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
        # CRITICAL: Shift to prevent look-ahead
        df['atr'] = true_range.ewm(span=window, adjust=False).mean().shift(1)
        return df
    
    def compute_all_features(self, df, corr_assets):
        print("\n" + "="*60)
        print("FEATURE ENGINEERING")
        print("="*60)
        
        df = self.compute_basic_returns(df)
        df['rolling_vol'] = df['log_ret'].rolling(self.config.VOL_LOOKBACK).std()
        df = self.compute_rolling_returns(df)
        df = self.compute_yang_zhang_volatility(df)
        df = self.compute_har_structure(df)
        df = self.compute_cross_correlations(df, corr_assets)
        df = self.compute_amihud_illiquidity(df)
        df = self.compute_momentum(df)
        df = self.compute_tsmom(df)
        df = self.compute_entropy(df)
        df = self.compute_hurst_exponent(df)
        df = self.compute_sma_features(df)
        df = self.compute_atr(df)
        
        print(f"[OK] Feature engineering complete. {len(df.columns)} columns.")
        return df


# ==============================================================================
# HMM REGIME DETECTION MODULE (V2 FIX: sklearn StandardScaler)
# ==============================================================================
class RegimeDetector:
    """
    HMM for regime detection.
    
    V2 FIX: Uses sklearn StandardScaler instead of manual implementation.
    V2 FIX: apply_shift parameter prevents look-ahead bias.
    """
    
    def __init__(self, config):
        self.config = config
        self.model = None
        self.scaler = None
    
    def prepare_regime_features(self, df):
        regime_features = []
        if "yzrv_d" in df.columns:
            regime_features.append("yzrv_d")
        if "yzrv_w" in df.columns:
            regime_features.append("yzrv_w")
        if "yzrv_m" in df.columns:
            regime_features.append("yzrv_m")
        corr_cols = [c for c in df.columns if c.startswith("corr_") and "_20" in c]
        regime_features.extend(corr_cols)
        return regime_features
    
    def fit(self, df, regime_features, apply_shift=True):
        """
        V2 FIX: apply_shift=True shifts features by 1 day to prevent look-ahead.
        """
        if apply_shift:
            X_df = df[regime_features].shift(1)
        else:
            X_df = df[regime_features]
        
        X = X_df.values
        valid_idx = ~np.isnan(X).any(axis=1)
        X_clean = X[valid_idx]
        
        if len(X_clean) < 100:
            raise ValueError("Insufficient clean data for HMM training")
        
        # V2 FIX: Use sklearn StandardScaler
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_clean)
        
        self.model = hmm.GaussianHMM(
            n_components=self.config.N_REGIMES,
            covariance_type=self.config.HMM_COVARIANCE,
            n_iter=self.config.HMM_N_ITER,
            random_state=self.config.HMM_RANDOM_STATE
        )
        self.model.fit(X_scaled)
        return self
    
    def predict(self, df, regime_features, apply_shift=True):
        if self.model is None:
            raise ValueError("Model not fitted")
        
        if apply_shift:
            X_df = df[regime_features].shift(1)
        else:
            X_df = df[regime_features]
        
        X = X_df.values
        valid_idx = ~np.isnan(X).any(axis=1)
        regimes = np.full(len(X), np.nan)
        
        if valid_idx.sum() == 0:
            return regimes
        
        X_valid = X[valid_idx]
        X_scaled = self.scaler.transform(X_valid)
        regimes_valid = self.model.predict(X_scaled)
        regimes[valid_idx] = regimes_valid
        return regimes


# ==============================================================================
# FEATURE SELECTION MODULE
# ==============================================================================
class FeatureSelector:
    """Dynamic feature selection to reduce overfitting."""
    
    def __init__(self, config):
        self.config = config
        self.selected_features = None
        self.feature_importance_df = None
    
    def select_features(self, X_train, y_train, feature_names, top_n=15):
        if isinstance(X_train, pd.DataFrame):
            actual_feature_names = list(X_train.columns)
            X_train_array = X_train.values
        else:
            actual_feature_names = feature_names
            X_train_array = X_train
        
        unique, counts = np.unique(y_train, return_counts=True)
        if len(unique) < 2:
            return actual_feature_names
        
        if X_train_array.shape[1] != len(actual_feature_names):
            return actual_feature_names
        
        rf = RandomForestClassifier(
            n_estimators=100, max_depth=3, min_samples_leaf=20,
            max_features='sqrt', random_state=42
        )
        
        try:
            rf.fit(X_train_array, y_train)
        except Exception:
            return actual_feature_names
        
        self.feature_importance_df = pd.DataFrame({
            'feature': actual_feature_names,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        self.selected_features = self.feature_importance_df.head(top_n)['feature'].tolist()
        
        print(f"\nFeature Selection: {len(actual_feature_names)} -> {len(self.selected_features)}")
        print(f"Top 5: {', '.join(self.feature_importance_df.head(5)['feature'].tolist())}")
        
        return self.selected_features
    
    def transform(self, X, feature_names=None):
        if self.selected_features is None:
            raise ValueError("Must call select_features() first")
        if isinstance(X, pd.DataFrame):
            return X[self.selected_features]
        if feature_names is None:
            raise ValueError("feature_names required for numpy array")
        indices = [feature_names.index(f) for f in self.selected_features if f in feature_names]
        return X[:, indices]


# ==============================================================================
# HYSTERESIS OPTIMIZER (V2)
# ==============================================================================
class HysteresisOptimizer:
    """
    Optimize entry/exit thresholds within each WFA fold.
    
    V2 FIX: Uses DAILY returns (not overlapping n-day) to avoid corrupted Sharpe.
    V2: Minimum holding period enforced to match prediction horizon.
    """
    
    def __init__(self, config):
        self.config = config
        self.entry_grid = config.ENTRY_THRESHOLD_GRID
        self.exit_grid = config.EXIT_THRESHOLD_GRID
        self.optimization_history = []
    
    def optimize_fold(self, probabilities, daily_returns, horizon=5):
        """
        Find optimal thresholds using DAILY returns.
        
        V3 UPDATE: Removed minimum holding constraint to allow T+1 exit (like V2).
        This provides more flexibility and faster reaction to changing signals.
        """
        best_sharpe = -np.inf
        best_params = (self.config.ENTRY_THRESHOLD, self.config.EXIT_THRESHOLD)
        results_grid = []
        
        for entry_thresh in self.entry_grid:
            for exit_thresh in self.exit_grid:
                if exit_thresh >= entry_thresh:
                    continue
                
                # V3 UPDATE: Use symmetric hysteresis WITHOUT holding constraint
                # Allows exit on T+1 if probability drops
                signals = self._apply_hysteresis_symmetric(
                    probabilities, entry_thresh, exit_thresh
                )
                
                if len(signals) < 10 or len(daily_returns) < 10:
                    continue
                
                min_len = min(len(signals), len(daily_returns))
                strat_returns = signals[:min_len] * daily_returns[:min_len]
                
                if len(strat_returns) > 10 and np.std(strat_returns) > 0:
                    sharpe = np.mean(strat_returns) / np.std(strat_returns) * np.sqrt(252)
                else:
                    sharpe = 0
                
                results_grid.append({
                    'entry': entry_thresh,
                    'exit': exit_thresh,
                    'sharpe': sharpe,
                    'n_trades': np.sum(np.abs(np.diff(signals)) > 0)
                })
                
                if sharpe > best_sharpe:
                    best_sharpe = sharpe
                    best_params = (entry_thresh, exit_thresh)
        
        self.optimization_history.append({
            'best_entry': best_params[0],
            'best_exit': best_params[1],
            'best_sharpe': best_sharpe
        })
        
        return best_params[0], best_params[1], best_sharpe
    
    def _apply_hysteresis_with_holding(self, probs, entry_thresh, exit_thresh, min_hold):
        """
        Apply hysteresis with minimum holding period.
        
        V2: Prevents excessive trading when horizon > 1.
        """
        n = len(probs)
        signals = np.zeros(n)
        position = 0
        days_held = 0
        
        short_entry_thresh = 1 - entry_thresh
        short_exit_thresh = 1 - exit_thresh
        
        for i in range(n):
            p = probs[i]
            
            if position == 0:
                if p > entry_thresh:
                    position = 1
                    days_held = 0
                elif p < short_entry_thresh:
                    position = -1
                    days_held = 0
            else:
                days_held += 1
                if days_held >= min_hold:
                    if position == 1:
                        if p < exit_thresh:
                            position = -1 if p < short_entry_thresh else 0
                            days_held = 0
                    elif position == -1:
                        if p > short_exit_thresh:
                            position = 1 if p > entry_thresh else 0
                            days_held = 0
            
            signals[i] = position
        
        return signals
    
    def _apply_hysteresis_symmetric(self, probs, entry_thresh, exit_thresh):
        """Apply symmetric hysteresis without holding constraint."""
        n = len(probs)
        signals = np.zeros(n)
        position = 0
        
        short_entry_thresh = 1 - entry_thresh
        short_exit_thresh = 1 - exit_thresh
        
        for i in range(n):
            p = probs[i]
            
            if position == 0:
                if p > entry_thresh:
                    position = 1
                elif p < short_entry_thresh:
                    position = -1
            elif position == 1:
                if p < exit_thresh:
                    position = -1 if p < short_entry_thresh else 0
            elif position == -1:
                if p > short_exit_thresh:
                    position = 1 if p > entry_thresh else 0
            
            signals[i] = position
        
        return signals
    
    def get_parameter_stability(self):
        """Calculate PSI (Parameter Stability Index)."""
        if len(self.optimization_history) < 2:
            return np.nan, np.nan, np.nan
        
        entries = [h['best_entry'] for h in self.optimization_history]
        exits = [h['best_exit'] for h in self.optimization_history]
        
        entry_cv = np.std(entries) / np.mean(entries) if np.mean(entries) > 0 else np.nan
        exit_cv = np.std(exits) / np.mean(exits) if np.mean(exits) > 0 else np.nan
        
        avg_cv = (entry_cv + exit_cv) / 2 if not np.isnan(entry_cv) else np.nan
        psi = 1 - min(avg_cv, 1) if not np.isnan(avg_cv) else np.nan
        
        return psi, entry_cv, exit_cv


# ==============================================================================
# ML MODEL WITH WFA AND HYSTERESIS OPTIMIZATION
# ==============================================================================
class MLTradingModel:
    """
    XGBoost with WFA and integrated hysteresis optimization.
    
    V2: Hysteresis optimized per fold on validation set.
    V2: 2x embargo for target leakage prevention.
    """
    
    def __init__(self, config):
        self.config = config
        self.models = []
        self.scalers = []
        self.feature_names = None
        self.feature_selector = FeatureSelector(config)
        self.hysteresis_optimizer = HysteresisOptimizer(config)
        self.fold_thresholds = []
    
    def prepare_ml_features(self, df):
        exclude_cols = [
            'target', 'target_return', 'target_return_raw', 'target_direction',
            'open', 'high', 'low', 'close', 'volume',
            'log_ret', 'ooc_ret', 'ooo_ret',
            'features_valid', 'ml_prediction', 'ml_probability', 'regime'
        ]
        exclude_patterns = ['_ret']
        
        feature_cols = []
        for c in df.columns:
            if c in exclude_cols:
                continue
            if any(p in c for p in exclude_patterns) and not any(
                x in c for x in ['roll', 'momentum', 'tsmom']):
                continue
            feature_cols.append(c)
        
        return feature_cols
    
    def create_target(self, df, horizon):
        df["target_return_raw"] = df["close"].shift(-horizon) / df["open"] - 1
        if self.config.VOL_NORMALIZATION:
            df["target_return"] = df["target_return_raw"] / (df['rolling_vol'] + 1e-8)
        else:
            df["target_return"] = df["target_return_raw"]
        df["target"] = (df["target_return"] > 0).astype(int)
        return df
    
    def walk_forward_validation(self, df):
        """
        WFA with hysteresis optimization.
        
        Structure per fold:
        - Train: 60% (for model)
        - Validate: 20% (for hysteresis optimization)
        - Test: 20% (true OOS)
        """
        print("\n" + "="*60)
        print("WALK-FORWARD VALIDATION WITH HYSTERESIS OPTIMIZATION")
        print("="*60)
        
        feature_cols = self.prepare_ml_features(df)
        self.feature_names = feature_cols
        
        # Shift features
        print("Applying temporal shift (T -> T-1)...")
        df[feature_cols] = df[feature_cols].shift(1)
        
        # Vol normalization
        # V3 FIX: rolling_vol is already shifted by 1 from the central shift above,
        # so using .shift(1) again would create a double-shift (2-day lag).
        # Use rolling_vol directly since it already represents T-1 values.
        if self.config.VOL_NORMALIZATION:
            return_features = [c for c in feature_cols if any(x in c for x in 
                ['ret', 'momentum', 'tsmom', 'ooc', 'ooo'])]
            for col in return_features:
                df[col] = df[col] / (df['rolling_vol'] + 1e-8)
        
        # Initialize regime detector
        regime_detector = RegimeDetector(self.config)
        regime_features = regime_detector.prepare_regime_features(df)
        df['regime'] = np.nan
        
        # Adaptive horizon
        initial_vol = df['rolling_vol'].iloc[:self.config.MIN_TRAIN_DAYS].median()
        annual_vol = initial_vol * np.sqrt(252) if pd.notna(initial_vol) else 0.3
        
        if annual_vol < 0.15:
            adjusted_horizon = self.config.PREDICTION_HORIZON * 2
            adjusted_refit = self.config.REFIT_FREQUENCY * 2
        else:
            adjusted_horizon = self.config.PREDICTION_HORIZON
            adjusted_refit = self.config.REFIT_FREQUENCY
        
        df = self.create_target(df, adjusted_horizon)
        embargo = adjusted_horizon * self.config.EMBARGO_MULTIPLIER
        
        valid_mask = df[feature_cols + ['target', 'target_return']].notna().all(axis=1)
        valid_data = df[valid_mask].copy()
        
        print(f"Valid data: {len(valid_data)} days")
        print(f"Embargo: {embargo} days")
        
        # Storage
        fold_results = []
        all_predictions = pd.Series(index=df.index, dtype=float)
        all_probabilities = pd.Series(index=df.index, dtype=float)
        all_thresholds = pd.DataFrame(index=df.index, columns=['entry_thresh', 'exit_thresh'])
        
        train_start_idx = 0
        train_end_idx = self.config.MIN_TRAIN_DAYS
        fold_counter = 0
        
        while train_end_idx < len(valid_data):
            fold_counter += 1
            safe_train_end = train_end_idx - embargo
            
            if safe_train_end <= train_start_idx + 50:
                train_end_idx += adjusted_refit
                continue
            
            # Split (V3 FIX: add embargo between val and test to prevent
            # validation target returns from overlapping with test period)
            val_test_embargo = adjusted_horizon  # gap = prediction horizon
            remaining = len(valid_data) - safe_train_end
            val_size = min(int(remaining * 0.5), adjusted_refit * 2)
            test_start_offset = val_size + val_test_embargo
            test_size = min(remaining - test_start_offset, adjusted_refit)
            
            if val_size < 10 or test_size < 5:
                train_end_idx += adjusted_refit
                continue
            
            val_end = safe_train_end + val_size
            test_start = safe_train_end + test_start_offset
            test_end = min(test_start + test_size, len(valid_data))
            
            train_dates = valid_data.index[train_start_idx:safe_train_end]
            val_dates = valid_data.index[safe_train_end:val_end]
            test_dates = valid_data.index[test_start:test_end]
            
            if len(train_dates) < 50 or len(val_dates) < 5 or len(test_dates) < 3:
                train_end_idx += adjusted_refit
                continue
            
            # Extract data
            X_train_df = valid_data.loc[train_dates, feature_cols].copy()
            y_train = valid_data.loc[train_dates, 'target'].values
            X_val_df = valid_data.loc[val_dates, feature_cols].copy()
            X_test_df = valid_data.loc[test_dates, feature_cols].copy()
            y_test = valid_data.loc[test_dates, 'target'].values
            
            # V2 FIX: Get DAILY returns for hysteresis optimization
            val_daily_returns = df.loc[val_dates, 'log_ret'].values
            
            # Class balance check
            unique_train, counts_train = np.unique(y_train, return_counts=True)
            if len(unique_train) < 2:
                train_end_idx += adjusted_refit
                continue
            
            minority_pct = counts_train.min() / len(y_train)
            if minority_pct < 0.02:
                train_end_idx += adjusted_refit
                continue
            
            # HMM (V3 FIX: apply_shift=False - features already centrally shifted)
            try:
                train_data_for_hmm = df.loc[train_dates]
                regime_detector.fit(train_data_for_hmm, regime_features, apply_shift=False)
                
                train_regimes = regime_detector.predict(train_data_for_hmm, regime_features, apply_shift=False)
                val_regimes = regime_detector.predict(df.loc[val_dates], regime_features, apply_shift=False)
                test_regimes = regime_detector.predict(df.loc[test_dates], regime_features, apply_shift=False)
                
                df.loc[train_dates, 'regime'] = train_regimes
                df.loc[val_dates, 'regime'] = val_regimes
                df.loc[test_dates, 'regime'] = test_regimes
                
                X_train_df['regime'] = np.nan_to_num(train_regimes, nan=1.0)
                X_val_df['regime'] = np.nan_to_num(val_regimes, nan=1.0)
                X_test_df['regime'] = np.nan_to_num(test_regimes, nan=1.0)
            except Exception:
                pass
            
            # Scale
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train_df).astype(np.float32)
            X_val = scaler.transform(X_val_df).astype(np.float32)
            X_test = scaler.transform(X_test_df).astype(np.float32)
            
            # Class weights
            if minority_pct < 0.3:
                class_weights = len(y_train) / (len(unique_train) * counts_train)
                sample_weights = np.array([class_weights[int(y)] for y in y_train])
            else:
                sample_weights = None
            
            # Train XGBoost
            model = xgb.XGBClassifier(**self.config.XGB_PARAMS)
            if sample_weights is not None:
                model.fit(X_train, y_train, sample_weight=sample_weights, verbose=False)
            else:
                model.fit(X_train, y_train, verbose=False)
            
            # Get validation probabilities
            val_probs = model.predict_proba(X_val)[:, 1]
            
            # Optimize hysteresis (V2: using daily returns)
            if self.config.OPTIMIZE_HYSTERESIS:
                opt_entry, opt_exit, val_sharpe = self.hysteresis_optimizer.optimize_fold(
                    val_probs, val_daily_returns, horizon=adjusted_horizon
                )
            else:
                opt_entry = self.config.ENTRY_THRESHOLD
                opt_exit = self.config.EXIT_THRESHOLD
                val_sharpe = np.nan
            
            self.fold_thresholds.append({
                'fold': fold_counter,
                'train_end': train_dates[-1],
                'entry_threshold': opt_entry,
                'exit_threshold': opt_exit,
                'val_sharpe': val_sharpe
            })
            
            # Apply to test - V3 UPDATE: Use symmetric without holding constraint
            test_probs = model.predict_proba(X_test)[:, 1]
            test_signals = self.hysteresis_optimizer._apply_hysteresis_symmetric(
                test_probs, opt_entry, opt_exit
            )
            
            test_pred = (test_probs > 0.5).astype(int)
            test_accuracy = accuracy_score(y_test, test_pred)
            
            # Store predictions
            for idx, (date, prob, sig) in enumerate(zip(test_dates, test_probs, test_signals)):
                all_predictions[date] = sig
                all_probabilities[date] = prob
                all_thresholds.loc[date, 'entry_thresh'] = opt_entry
                all_thresholds.loc[date, 'exit_thresh'] = opt_exit
            
            self.models.append(model)
            self.scalers.append(scaler)
            
            if fold_counter % 10 == 1:
                print(f"Fold {fold_counter}: Train {len(train_dates)}, Val {len(val_dates)}, Test {len(test_dates)}")
                print(f"  Acc: {test_accuracy:.4f}, Entry: {opt_entry:.2f}, Exit: {opt_exit:.2f}")
            
            fold_results.append({
                'fold': fold_counter,
                'train_end': train_dates[-1],
                'test_end': test_dates[-1],
                'accuracy': test_accuracy,
                'val_sharpe': val_sharpe,
                'entry_threshold': opt_entry,
                'exit_threshold': opt_exit
            })
            
            train_end_idx += adjusted_refit
        
        results_df = pd.DataFrame(fold_results)
        
        print(f"\nWFA Summary: {len(results_df)} folds")
        print(f"Avg accuracy: {results_df['accuracy'].mean():.4f}")
        
        # PSI
        psi, entry_cv, exit_cv = self.hysteresis_optimizer.get_parameter_stability()
        print(f"PSI: {psi:.4f} (Entry CV: {entry_cv:.4f}, Exit CV: {exit_cv:.4f})")
        
        df['ml_prediction'] = all_predictions
        df['ml_probability'] = all_probabilities
        df['entry_thresh'] = all_thresholds['entry_thresh']
        df['exit_thresh'] = all_thresholds['exit_thresh']
        df.attrs['psi'] = psi
        
        return df, results_df


# ==============================================================================
# BACKTESTING ENGINE (V2 FIXES)
# ==============================================================================
class HybridBacktester:
    """
    Production-grade backtester.
    
    V2 FIXES:
    - Cash reserve properly updated in non-compounding mode
    - CVaR stop handles both long AND short correctly
    - Volatility scaling, regime allocation, probability sizing
    """
    
    def __init__(self, config):
        self.config = config
        self.trades = []
        self.daily_stats = []
    
    def run(self, df, df_benchmark):
        print("\n" + "="*60)
        print("RUNNING BACKTEST")
        print("="*60)
        
        start_date = pd.to_datetime(self.config.BACKTEST_START)
        end_date = pd.to_datetime(self.config.BACKTEST_END)
        
        df_bt = df[(df.index >= start_date) & (df.index <= end_date)].copy()
        df_bm = df_benchmark[(df_benchmark.index >= start_date) & 
                             (df_benchmark.index <= end_date)].copy()
        
        if len(df_bt) == 0:
            raise ValueError("No data in backtest period")
        
        # Adjust start if needed
        first_pred_date = df_bt[df_bt['ml_prediction'].notna()].index.min()
        if pd.notna(first_pred_date) and first_pred_date > start_date:
            df_bt = df_bt[df_bt.index >= first_pred_date]
        
        print(f"Period: {df_bt.index[0].date()} to {df_bt.index[-1].date()}")
        print(f"Trading days: {len(df_bt)}")
        
        # Initialize
        capital = self.config.INITIAL_CAPITAL
        cash = capital
        position = 0
        position_value = 0
        prev_signal = 0
        entry_price = None
        entry_date = None
        
        portfolio_values = []
        strategy_returns_history = []
        
        df_bm['bm_return'] = df_bm['close'].pct_change()
        benchmark_values = [capital]
        
        for i in range(len(df_bt)):
            date = df_bt.index[i]
            row = df_bt.iloc[i]
            
            ml_signal = row['ml_prediction'] if pd.notna(row['ml_prediction']) else 0
            ml_prob = row['ml_probability'] if pd.notna(row['ml_probability']) else 0.5
            
            entry_thresh = row.get('entry_thresh', self.config.ENTRY_THRESHOLD)
            exit_thresh = row.get('exit_thresh', self.config.EXIT_THRESHOLD)
            if pd.isna(entry_thresh):
                entry_thresh = self.config.ENTRY_THRESHOLD
            if pd.isna(exit_thresh):
                exit_thresh = self.config.EXIT_THRESHOLD
            
            # Apply hysteresis (V2: symmetric)
            short_entry_thresh = 1 - entry_thresh
            short_exit_thresh = 1 - exit_thresh
            
            if position == 0:
                if ml_prob > entry_thresh:
                    target_direction = 1
                elif ml_prob < short_entry_thresh:
                    target_direction = -1
                else:
                    target_direction = 0
            elif position > 0:
                if ml_prob < exit_thresh:
                    target_direction = -1 if ml_prob < short_entry_thresh else 0
                else:
                    target_direction = 1
            else:
                if ml_prob > short_exit_thresh:
                    target_direction = 1 if ml_prob > entry_thresh else 0
                else:
                    target_direction = -1
            
            # Chandelier stop (V3 FIX: use prev_close to avoid look-ahead)
            chandelier_triggered = False
            if self.config.USE_CHANDELIER_STOP and position != 0 and entry_date is not None and i > 0:
                entry_idx = df_bt.index.get_loc(entry_date)
                current_atr = row['atr'] if pd.notna(row['atr']) else (row['high'] - row['low'])
                prev_close_for_stop = df_bt.iloc[i-1]['close']
                
                if position > 0:
                    highest_high = df_bt.iloc[entry_idx:i]['high'].max()
                    chandelier_stop = highest_high - (self.config.ATR_MULTIPLIER * current_atr)
                    if prev_close_for_stop < chandelier_stop:
                        target_direction = 0
                        chandelier_triggered = True
                elif position < 0:
                    lowest_low = df_bt.iloc[entry_idx:i]['low'].min()
                    chandelier_stop = lowest_low + (self.config.ATR_MULTIPLIER * current_atr)
                    if prev_close_for_stop > chandelier_stop:
                        target_direction = 0
                        chandelier_triggered = True
            
            # CVaR stop (V3 FIX: use prev_close to avoid look-ahead)
            cvar_triggered = False
            if self.config.USE_CVAR_STOP and position != 0 and entry_price is not None and i > 0:
                if len(strategy_returns_history) >= self.config.CVAR_LOOKBACK:
                    lookback_returns = pd.Series(strategy_returns_history[-self.config.CVAR_LOOKBACK:])
                    var_threshold = lookback_returns.quantile(1 - self.config.CVAR_QUANTILE)
                    tail_returns = lookback_returns[lookback_returns <= var_threshold]
                    
                    if len(tail_returns) > 0:
                        cvar = tail_returns.mean()
                        stop_threshold = cvar * self.config.CVAR_MULTIPLIER
                        
                        prev_close_for_cvar = df_bt.iloc[i-1]['close']
                        # V3 FIX: Use prev_close (known at signal time) not row['close']
                        if position > 0:
                            position_return = (prev_close_for_cvar / entry_price) - 1
                        else:
                            position_return = (entry_price / prev_close_for_cvar) - 1
                        
                        if position_return < stop_threshold:
                            target_direction = 0
                            cvar_triggered = True
            
            # Position sizing
            execute_trade = (i == 0) or (target_direction != prev_signal)
            
            if execute_trade and target_direction != 0:
                size_price = df_bt.iloc[i-1]['close'] if i > 0 else row['close']
                
                if self.config.COMPOUNDING_MODE:
                    portfolio_value_for_sizing = cash + position_value
                else:
                    portfolio_value_for_sizing = capital
                
                target_notional = portfolio_value_for_sizing * self.config.LEVERAGE
                
                # V2: Volatility scaling
                if self.config.USE_VOL_SCALING:
                    current_vol = row['rolling_vol'] if pd.notna(row['rolling_vol']) else 0.02
                    annual_vol = current_vol * np.sqrt(252)
                    if annual_vol > 0:
                        vol_scalar = self.config.TARGET_VOLATILITY / annual_vol
                        vol_scalar = min(max(vol_scalar, 0.2), self.config.VOL_SCALE_CAP)
                        target_notional *= vol_scalar
                
                # V2: Regime allocation
                if self.config.USE_REGIME_ALLOCATION:
                    regime = int(row['regime']) if pd.notna(row['regime']) else 1
                    regime_mult = self.config.REGIME_MULTIPLIERS.get(regime, 0.7)
                    target_notional *= regime_mult
                
                # V2: Probability sizing
                if self.config.USE_PROBABILITY_SIZING:
                    confidence = abs(ml_prob - 0.5) * 2
                    prob_mult = self.config.MIN_CONFIDENCE_ALLOCATION + (
                        (1 - self.config.MIN_CONFIDENCE_ALLOCATION) * confidence
                    )
                    target_notional *= prob_mult
                
                target_notional *= target_direction
                target_shares = target_notional / size_price
            elif execute_trade and target_direction == 0:
                target_shares = 0
            else:
                target_shares = position
            
            # Execute trade
            old_position = position
            old_position_value = position_value
            
            if execute_trade:
                trade_shares = target_shares - old_position
                
                if abs(trade_shares) > 0.001:
                    execution_price = row['open']
                    trade_value = abs(trade_shares * execution_price)
                    
                    commission = trade_value * self.config.COMMISSION_PCT
                    slippage = trade_value * (self.config.SLIPPAGE_BPS / 10000)
                    total_costs = commission + slippage
                    
                    if trade_shares > 0:
                        cash -= trade_shares * execution_price + total_costs
                    else:
                        cash += abs(trade_shares * execution_price) - total_costs
                    
                    position = target_shares
                    
                    if position != 0:
                        entry_price = execution_price
                        entry_date = date
                    else:
                        entry_price = None
                        entry_date = None
                    
                    self.trades.append({
                        'date': date,
                        'direction': target_direction,
                        'shares': trade_shares,
                        'price': execution_price,
                        'value': trade_value,
                        'commission': commission,
                        'slippage': slippage,
                        'total_cost': total_costs,
                        'chandelier_stop': chandelier_triggered,
                        'cvar_stop': cvar_triggered,
                        'entry_thresh': entry_thresh,
                        'exit_thresh': exit_thresh
                    })
            
            # P&L calculation
            if i > 0:
                prev_close = df_bt.iloc[i-1]['close']
                
                if execute_trade and abs(target_shares - old_position) > 0.001:
                    gap_return = (row['open'] / prev_close) - 1
                    gap_pnl = old_position_value * gap_return
                    intraday_return = (row['close'] / row['open']) - 1
                    intraday_pnl = (position * row['open']) * intraday_return
                    position_pnl = gap_pnl + intraday_pnl
                else:
                    pnl_return = (row['close'] / prev_close) - 1
                    position_pnl = position_value * pnl_return
                
                position_value = position * row['close'] if position != 0 else 0
                
                # V2 FIX: Update cash with P&L in non-compounding mode
                if not self.config.COMPOUNDING_MODE:
                    cash += position_pnl
                
                if portfolio_values and portfolio_values[-1] > 0:
                    portfolio_return = position_pnl / portfolio_values[-1]
                else:
                    portfolio_return = 0
                
                strategy_returns_history.append(portfolio_return)
            else:
                portfolio_return = 0
                position_pnl = 0
                position_value = position * row['close'] if position != 0 else 0
            
            portfolio_value = cash + position_value
            portfolio_values.append(portfolio_value)
            
            # Benchmark
            if date in df_bm.index:
                bm_ret = df_bm.loc[date, 'bm_return']
                if pd.notna(bm_ret):
                    benchmark_values.append(benchmark_values[-1] * (1 + bm_ret))
                else:
                    benchmark_values.append(benchmark_values[-1])
            else:
                benchmark_values.append(benchmark_values[-1])
            
            self.daily_stats.append({
                'date': date,
                'portfolio_value': portfolio_value,
                'cash': cash,
                'position_value': position_value,
                'position_shares': position,
                'signal': target_direction,
                'ml_probability': ml_prob,
                'entry_thresh': entry_thresh,
                'exit_thresh': exit_thresh,
                'execute_trade': execute_trade,
                'portfolio_return': portfolio_return,
                'benchmark_value': benchmark_values[-1]
            })
            
            prev_signal = target_direction
        
        results_df = pd.DataFrame(self.daily_stats).set_index('date')
        results_df['benchmark_return'] = results_df['benchmark_value'].pct_change()
        results_df['cumulative_return'] = results_df['portfolio_value'] / capital - 1
        results_df['cumulative_benchmark'] = results_df['benchmark_value'] / capital - 1
        
        print(f"\n[OK] Backtest complete")
        print(f"Trades: {len(self.trades)}")
        print(f"Final: ${results_df['portfolio_value'].iloc[-1]:,.2f}")
        print(f"Return: {results_df['cumulative_return'].iloc[-1]*100:.2f}%")
        
        return results_df
    
    def get_trades_df(self):
        if not self.trades:
            return pd.DataFrame()
        return pd.DataFrame(self.trades).set_index('date')


# ==============================================================================
# ROBUSTNESS METRICS (V2: PSI integrated)
# ==============================================================================
class RobustnessMetrics:
    """Comprehensive robustness metrics with PSI integration."""
    
    def __init__(self, config):
        self.config = config
    
    def deflated_sharpe_ratio(self, returns, n_tests=None):
        if n_tests is None:
            n_tests = self.config.N_TESTS_ADJUSTMENT
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        sr = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        skew = returns_clean.skew()
        kurt = returns_clean.kurtosis()
        adjustment = np.sqrt(1 + (n_tests - 1) * (skew**2 + kurt/4))
        return sr / adjustment if adjustment > 0 else np.nan
    
    def probabilistic_sharpe_ratio(self, returns, benchmark_sr=0):
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        sr_obs = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        n = len(returns_clean)
        skew = returns_clean.skew()
        kurt = returns_clean.kurtosis()
        sr_std = np.sqrt((1 + 0.5 * sr_obs**2 - skew * sr_obs + (kurt - 3) / 4 * sr_obs**2) / n)
        test_stat = (sr_obs - benchmark_sr) / sr_std
        return t.cdf(test_stat, df=n-1)
    
    def walk_forward_efficiency(self, is_sharpe, oos_sharpe):
        if is_sharpe == 0 or np.isnan(is_sharpe) or np.isnan(oos_sharpe):
            return np.nan
        return oos_sharpe / is_sharpe
    
    def hurst_exponent(self, returns):
        returns_clean = returns.dropna()
        if len(returns_clean) < 100:
            return np.nan
        lags = range(2, min(100, len(returns_clean) // 2))
        rs_values = []
        for lag in lags:
            n_blocks = len(returns_clean) // lag
            if n_blocks < 2:
                continue
            rs_lag = []
            for i in range(n_blocks):
                block = returns_clean.iloc[i*lag:(i+1)*lag]
                mean_adj = block - block.mean()
                cum_sum = mean_adj.cumsum()
                R = cum_sum.max() - cum_sum.min()
                S = block.std()
                if S > 0:
                    rs_lag.append(R / S)
            if rs_lag:
                rs_values.append((lag, np.mean(rs_lag)))
        if len(rs_values) < 5:
            return np.nan
        lags_arr = np.array([x[0] for x in rs_values])
        rs_arr = np.array([x[1] for x in rs_values])
        return np.polyfit(np.log(lags_arr), np.log(rs_arr), 1)[0]
    
    def conditional_drawdown_at_risk(self, returns, quantile=None):
        if quantile is None:
            quantile = self.config.CDAR_QUANTILE
        returns_clean = returns.dropna()
        if len(returns_clean) < 30:
            return np.nan
        cum_returns = (1 + returns_clean).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        threshold = drawdown.quantile(1 - quantile)
        tail_drawdowns = drawdown[drawdown <= threshold]
        if len(tail_drawdowns) == 0:
            return np.nan
        return abs(tail_drawdowns.mean())
    
    def global_robustness_index(self, returns, is_sharpe=None, oos_sharpe=None, psi=None):
        """V2: PSI integrated into GRI calculation."""
        dsr = self.deflated_sharpe_ratio(returns)
        psr = self.probabilistic_sharpe_ratio(returns)
        hurst = self.hurst_exponent(returns)
        cdar = self.conditional_drawdown_at_risk(returns)
        
        if is_sharpe and oos_sharpe:
            wfe = self.walk_forward_efficiency(is_sharpe, oos_sharpe)
        else:
            wfe = np.nan
        
        scores = []
        if not np.isnan(dsr):
            scores.append(min(max(dsr / 2, 0), 1))
        if not np.isnan(psr):
            scores.append(psr)
        if not np.isnan(wfe):
            scores.append(min(max(wfe, 0), 1))
        if not np.isnan(hurst):
            scores.append(max(1 - 2 * abs(hurst - 0.5), 0))
        if not np.isnan(cdar):
            scores.append(1 - min(abs(cdar) / 0.5, 1))
        if psi is not None and not np.isnan(psi):
            scores.append(psi)
        
        return np.mean(scores) if scores else np.nan
    
    def compute_all_metrics(self, returns, predictions=None, realized_returns=None,
                           is_sharpe=None, oos_sharpe=None, psi=None):
        metrics = {}
        returns_clean = returns.dropna()
        
        metrics['sharpe_ratio'] = returns_clean.mean() / returns_clean.std() * np.sqrt(252)
        metrics['annual_return'] = returns_clean.mean() * 252
        metrics['annual_volatility'] = returns_clean.std() * np.sqrt(252)
        
        cum_returns = (1 + returns_clean).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        metrics['max_drawdown'] = drawdown.min()
        
        metrics['deflated_sharpe'] = self.deflated_sharpe_ratio(returns)
        metrics['probabilistic_sharpe'] = self.probabilistic_sharpe_ratio(returns)
        metrics['hurst_exponent'] = self.hurst_exponent(returns)
        metrics['cdar'] = self.conditional_drawdown_at_risk(returns)
        
        if is_sharpe and oos_sharpe:
            metrics['walk_forward_efficiency'] = self.walk_forward_efficiency(is_sharpe, oos_sharpe)
        else:
            metrics['walk_forward_efficiency'] = np.nan
        
        metrics['psi'] = psi if psi is not None else np.nan
        metrics['global_robustness_index'] = self.global_robustness_index(
            returns, is_sharpe, oos_sharpe, psi
        )
        
        print(f"\nSharpe: {metrics['sharpe_ratio']:.4f}")
        print(f"Annual Return: {metrics['annual_return']*100:.2f}%")
        print(f"Max DD: {metrics['max_drawdown']*100:.2f}%")
        print(f"DSR: {metrics['deflated_sharpe']:.4f}")
        print(f"PSR: {metrics['probabilistic_sharpe']:.4f}")
        print(f"PSI: {metrics['psi']:.4f}")
        print(f"GRI: {metrics['global_robustness_index']:.4f}")
        
        return metrics


# ==============================================================================
# MULTI-TICKER VALIDATION MODULE (PRESERVED)
# ==============================================================================
class MultiTickerValidator:
    """Cross-validation across multiple assets."""
    
    def __init__(self, base_config):
        self.base_config = base_config
        self.results = {}
        self.metrics_summary = None
    
    def validate_ticker(self, ticker, config_overrides=None):
        print(f"\n{'='*70}")
        print(f"VALIDATING: {ticker}")
        print('='*70)
        
        config = StrategyConfig()
        config.SYMBOL_MAIN = ticker
        
        if config_overrides:
            for key, value in config_overrides.items():
                setattr(config, key, value)
        
        try:
            feature_engine = FeatureEngine(config)
            ml_model = MLTradingModel(config)
            backtester = HybridBacktester(config)
            metrics_calc = RobustnessMetrics(config)
            
            df_main = fetch_ohlcv(ticker, config.START_DATE, config.END_DATE)
            
            if len(df_main) < 300:
                return {'status': 'failed', 'error': f'Insufficient data: {len(df_main)} days'}
            
            corr_assets = {}
            for symbol in config.SYMBOLS_CORR:
                df_corr = fetch_ohlcv(symbol, config.START_DATE, config.END_DATE)
                if not df_corr.empty:
                    corr_assets[symbol.lower()] = df_corr
            
            df_benchmark = fetch_ohlcv(config.BENCHMARK, config.START_DATE, config.END_DATE)
            
            df_features = feature_engine.compute_all_features(df_main, corr_assets)
            df_ml, wf_results = ml_model.walk_forward_validation(df_features)
            
            psi = df_ml.attrs.get('psi', np.nan)
            
            results_df = backtester.run(df_ml, df_benchmark)
            
            mid_point = len(results_df) // 2
            is_returns = results_df['portfolio_return'].iloc[:mid_point]
            oos_returns = results_df['portfolio_return'].iloc[mid_point:]
            
            is_sharpe = is_returns.mean() / is_returns.std() * np.sqrt(252) if is_returns.std() > 0 else np.nan
            oos_sharpe = oos_returns.mean() / oos_returns.std() * np.sqrt(252) if oos_returns.std() > 0 else np.nan
            
            robustness = metrics_calc.compute_all_metrics(
                results_df['portfolio_return'],
                is_sharpe=is_sharpe,
                oos_sharpe=oos_sharpe,
                psi=psi
            )
            
            result = {
                'status': 'success',
                'ticker': ticker,
                'total_return': results_df['cumulative_return'].iloc[-1],
                'sharpe_ratio': robustness['sharpe_ratio'],
                'max_drawdown': robustness['max_drawdown'],
                'deflated_sharpe': robustness['deflated_sharpe'],
                'probabilistic_sharpe': robustness['probabilistic_sharpe'],
                'walk_forward_efficiency': robustness['walk_forward_efficiency'],
                'hurst_exponent': robustness['hurst_exponent'],
                'cdar': robustness['cdar'],
                'psi': psi,
                'global_robustness_index': robustness['global_robustness_index'],
                'total_trades': len(backtester.trades),
                'trading_days': len(results_df)
            }
            
            print(f"\n[OK] {ticker}: GRI={result['global_robustness_index']:.4f}, SR={result['sharpe_ratio']:.4f}")
            return result
            
        except Exception as e:
            print(f"[ERROR] {ticker}: {str(e)}")
            return {'status': 'failed', 'ticker': ticker, 'error': str(e)}
    
    def run_validation(self, tickers, config_overrides=None, gri_threshold=0.60):
        print("\n" + "="*70)
        print("MULTI-TICKER VALIDATION")
        print("="*70)
        print(f"Tickers: {', '.join(tickers)}")
        print(f"GRI threshold: {gri_threshold}")
        
        for ticker in tickers:
            result = self.validate_ticker(ticker, config_overrides)
            self.results[ticker] = result
        
        summary_data = []
        for ticker, result in self.results.items():
            if result['status'] == 'success':
                summary_data.append({
                    'Ticker': ticker,
                    'Return (%)': result['total_return'] * 100,
                    'Sharpe': result['sharpe_ratio'],
                    'Max DD (%)': result['max_drawdown'] * 100,
                    'DSR': result['deflated_sharpe'],
                    'PSR': result['probabilistic_sharpe'],
                    'WFE': result['walk_forward_efficiency'],
                    'PSI': result['psi'],
                    'GRI': result['global_robustness_index'],
                    'Pass': 'Y' if result['global_robustness_index'] > gri_threshold else 'N',
                    'Trades': result['total_trades']
                })
            else:
                summary_data.append({
                    'Ticker': ticker, 'Return (%)': np.nan, 'Sharpe': np.nan,
                    'Max DD (%)': np.nan, 'DSR': np.nan, 'PSR': np.nan,
                    'WFE': np.nan, 'PSI': np.nan, 'GRI': np.nan, 'Pass': 'N', 'Trades': 0
                })
        
        self.metrics_summary = pd.DataFrame(summary_data)
        
        successful = self.metrics_summary[self.metrics_summary['GRI'] > gri_threshold]
        success_rate = len(successful) / len(tickers)
        
        print("\n" + "="*70)
        print("VALIDATION SUMMARY")
        print("="*70)
        print(self.metrics_summary.to_string(index=False))
        print(f"\nSUCCESS RATE: {success_rate*100:.1f}% ({len(successful)}/{len(tickers)})")
        
        if success_rate >= 0.70:
            print("[OK] STRATEGY IS ROBUST")
        elif success_rate >= 0.50:
            print("[WARN] STRATEGY IS MARGINAL")
        else:
            print("[FAIL] STRATEGY OVERFITS")
        
        return self.metrics_summary
    
    def save_results(self, output_dir='results/xgboost-regime-adaptive'):
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        if self.metrics_summary is None:
            return
        
        self.metrics_summary.to_csv(f'{output_dir}/multi_ticker_validation_v3.csv', index=False)
        with open(f'{output_dir}/multi_ticker_detailed_v3.json', 'w') as f:
            json.dump(self.results, f, indent=2, default=str)
        print(f"[OK] Results saved to {output_dir}")


# ==============================================================================
# VISUALIZATION
# ==============================================================================
def create_performance_report(results_df, config, fold_thresholds=None):
    fig, axes = plt.subplots(4, 2, figsize=config.PLOT_FIGSIZE, dpi=config.PLOT_DPI)
    fig.suptitle(f'ML Regime Strategy V3: {config.SYMBOL_MAIN}', fontsize=16, fontweight='bold')
    
    # Cumulative Returns
    ax = axes[0, 0]
    ax.plot(results_df.index, results_df['cumulative_return'] * 100, label='Strategy', linewidth=2)
    ax.plot(results_df.index, results_df['cumulative_benchmark'] * 100, 
            label=f'Benchmark ({config.BENCHMARK})', linewidth=2, alpha=0.7)
    ax.set_title('Cumulative Returns')
    ax.set_ylabel('Return (%)')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Drawdown
    ax = axes[0, 1]
    cum_ret = (1 + results_df['portfolio_return']).cumprod()
    running_max = cum_ret.expanding().max()
    drawdown = (cum_ret - running_max) / running_max * 100
    ax.fill_between(drawdown.index, 0, drawdown, alpha=0.3, color='red')
    ax.set_title('Drawdown')
    ax.set_ylabel('Drawdown (%)')
    ax.grid(alpha=0.3)
    
    # Rolling Sharpe
    ax = axes[1, 0]
    rolling_sharpe = results_df['portfolio_return'].rolling(60).mean() / results_df['portfolio_return'].rolling(60).std() * np.sqrt(252)
    ax.plot(rolling_sharpe.index, rolling_sharpe, linewidth=1.5)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.set_title('Rolling 60-Day Sharpe')
    ax.grid(alpha=0.3)
    
    # Position Distribution
    ax = axes[1, 1]
    signal_counts = results_df['signal'].value_counts().sort_index()
    colors = {-1: 'red', 0: 'gray', 1: 'green'}
    ax.bar(signal_counts.index, signal_counts.values, color=[colors.get(x, 'blue') for x in signal_counts.index])
    ax.set_title('Position Distribution')
    ax.grid(alpha=0.3)
    
    # Monthly Returns
    ax = axes[2, 0]
    monthly_returns = results_df['portfolio_return'].resample('ME').apply(lambda x: (1 + x).prod() - 1) * 100
    monthly_pivot = monthly_returns.to_frame('return')
    monthly_pivot['year'] = monthly_pivot.index.year
    monthly_pivot['month'] = monthly_pivot.index.month
    pivot = monthly_pivot.pivot(index='year', columns='month', values='return')
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
    ax.set_xticks(range(12))
    ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('Monthly Returns (%)')
    plt.colorbar(im, ax=ax)
    
    # Return Distribution
    ax = axes[2, 1]
    ax.hist(results_df['portfolio_return'] * 100, bins=50, alpha=0.7, edgecolor='black')
    ax.axvline(x=results_df['portfolio_return'].mean() * 100, color='red', linestyle='--', linewidth=2)
    ax.set_title('Daily Returns')
    ax.grid(alpha=0.3)
    
    # Threshold Evolution
    ax = axes[3, 0]
    if fold_thresholds and len(fold_thresholds) > 0:
        fold_nums = [f['fold'] for f in fold_thresholds]
        entries = [f['entry_threshold'] for f in fold_thresholds]
        exits = [f['exit_threshold'] for f in fold_thresholds]
        ax.plot(fold_nums, entries, 'b-o', label='Entry', markersize=3)
        ax.plot(fold_nums, exits, 'r-o', label='Exit', markersize=3)
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
        ax.set_title('Threshold Evolution')
        ax.legend()
        ax.grid(alpha=0.3)
    
    # Probability vs Signal
    ax = axes[3, 1]
    if 'ml_probability' in results_df.columns:
        sample_size = min(500, len(results_df))
        sample_idx = np.random.choice(len(results_df), sample_size, replace=False)
        sample_idx.sort()
        probs = results_df['ml_probability'].iloc[sample_idx]
        signals = results_df['signal'].iloc[sample_idx]
        colors_map = {-1: 'red', 0: 'gray', 1: 'green'}
        c = [colors_map.get(s, 'blue') for s in signals]
        ax.scatter(range(len(probs)), probs, c=c, alpha=0.5, s=10)
        ax.axhline(y=0.5, color='black', linestyle='-', alpha=0.3)
        ax.set_title('Probability vs Signal')
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    return fig


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
def main():
    """Run strategy on main ticker."""
    print("="*60)
    print("ML REGIME-ADAPTIVE TRADING STRATEGY V3")
    print("="*60)
    
    config = StrategyConfig()
    
    print(f"\nConfig: {config.SYMBOL_MAIN}, {config.BACKTEST_START} to {config.BACKTEST_END}")
    print(f"Hysteresis Optimization: {config.OPTIMIZE_HYSTERESIS}")
    print(f"Vol Scaling: {config.USE_VOL_SCALING}")
    print(f"Regime Allocation: {config.USE_REGIME_ALLOCATION}")
    
    feature_engine = FeatureEngine(config)
    ml_model = MLTradingModel(config)
    backtester = HybridBacktester(config)
    metrics_calc = RobustnessMetrics(config)
    
    # Fetch data
    df_main = fetch_ohlcv(config.SYMBOL_MAIN, config.START_DATE, config.END_DATE)
    if df_main.empty:
        raise ValueError(f"Failed to fetch {config.SYMBOL_MAIN}")
    
    corr_assets = {}
    for symbol in config.SYMBOLS_CORR:
        df_corr = fetch_ohlcv(symbol, config.START_DATE, config.END_DATE)
        if not df_corr.empty:
            corr_assets[symbol.lower()] = df_corr
    
    df_benchmark = fetch_ohlcv(config.BENCHMARK, config.START_DATE, config.END_DATE)
    
    # Feature engineering
    df_features = feature_engine.compute_all_features(df_main, corr_assets)
    
    # WFA with hysteresis optimization
    df_ml, wf_results = ml_model.walk_forward_validation(df_features)
    psi = df_ml.attrs.get('psi', np.nan)
    
    # Backtest
    results_df = backtester.run(df_ml, df_benchmark)
    trades_df = backtester.get_trades_df()
    
    # Metrics
    mid_point = len(results_df) // 2
    is_returns = results_df['portfolio_return'].iloc[:mid_point]
    oos_returns = results_df['portfolio_return'].iloc[mid_point:]
    is_sharpe = is_returns.mean() / is_returns.std() * np.sqrt(252) if is_returns.std() > 0 else np.nan
    oos_sharpe = oos_returns.mean() / oos_returns.std() * np.sqrt(252) if oos_returns.std() > 0 else np.nan
    
    robustness_metrics = metrics_calc.compute_all_metrics(
        results_df['portfolio_return'],
        is_sharpe=is_sharpe,
        oos_sharpe=oos_sharpe,
        psi=psi
    )
    
    # Visualization
    fig = create_performance_report(results_df, config, ml_model.fold_thresholds)
    
    # Save
    output_dir = 'results/xgboost-regime-adaptive'
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    fig.savefig(f'{output_dir}/ml_regime_strategy_v3.png', dpi=config.PLOT_DPI, bbox_inches='tight')
    results_df.to_csv(f'{output_dir}/backtest_results_v3.csv')
    if not trades_df.empty:
        trades_df.to_csv(f'{output_dir}/trades_v3.csv')
    wf_results.to_csv(f'{output_dir}/walk_forward_results_v3.csv', index=False)
    
    thresholds_df = pd.DataFrame(ml_model.fold_thresholds)
    thresholds_df.to_csv(f'{output_dir}/threshold_evolution_v3.csv', index=False)
    
    pd.DataFrame([robustness_metrics]).to_csv(f'{output_dir}/robustness_metrics_v3.csv', index=False)
    
    print("\n" + "="*60)
    print("COMPLETE")
    print("="*60)
    print(f"Return: {results_df['cumulative_return'].iloc[-1]*100:.2f}%")
    print(f"Sharpe: {robustness_metrics['sharpe_ratio']:.4f}")
    print(f"Max DD: {robustness_metrics['max_drawdown']*100:.2f}%")
    print(f"PSI: {psi:.4f}")
    print(f"GRI: {robustness_metrics['global_robustness_index']:.4f}")
    
    return results_df, trades_df, robustness_metrics


def run_multi_ticker_validation():
    """Run validation across all tickers (PRESERVED)."""
    test_tickers = [
        'NVDA', 'COIN', 'MSTR', 'TSLA', 'AMZN', 'AMD', 'NFLX', 
        'INTC', 'QCOM', 'MU', 'MARA', 'GOOGL', 'META', 'RIOT'
    ]
    
    validator = MultiTickerValidator(StrategyConfig())
    summary = validator.run_validation(tickers=test_tickers, gri_threshold=0.60)
    validator.save_results()
    
    return summary


# ==============================================================================
# ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    results, trades, metrics = main()
    
    # Run the additional validation across the configured ticker list.
    multi_ticker_summary = run_multi_ticker_validation()
